In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
#| label: smartwatch-contract-audit
import json
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("..")
RESULT_JSON = (
    ROOT / "results/comprehensive_latest_48_models/"
           "smartwatch/four_device_results.json"
)
SUMMARY_CSV = (
    ROOT / "results/comprehensive_latest_48_models/"
           "tables/smartwatch_four_device_summary.csv"
)
payload = json.loads(RESULT_JSON.read_text())
watch = pd.read_csv(SUMMARY_CSV)
protocol = payload["protocol"]

pd.DataFrame({
    "quantity": [
        "summary rows", "reconstruction models", "devices",
        "expected rows", "task", "evaluation sample rate",
        "evaluation length", "human diagnostic ground truth"
    ],
    "value": [
        len(watch), watch.model_id.nunique(), watch.device.nunique(),
        watch.model_id.nunique() * watch.device.nunique(),
        protocol["task"], protocol["evaluation_sample_rate_hz"],
        protocol["evaluation_length_samples"],
        protocol["evaluation_taxonomy"]["human_diagnostic_ground_truth"],
    ]
})

,quantity,value
0,summary rows,192
1,reconstruction models,48
2,devices,4
3,expected rows,192
4,task,lead_II_to_eleven_missing_leads_zero_shot_OOD
5,evaluation sample rate,500
6,evaluation length,5000
7,human diagnostic ground truth,unavailable


In [3]:
#| label: smartwatch-pairing-audit
#| tbl-cap: Watch-to-reference pairing audit from the locked protocol.
pairing_rows = []
for device, audit in protocol["pairing_audit"].items():
    pairing_rows.append({
        "device": device,
        "discovered_watch_records": audit["discovered_watch_records"],
        "paired_records": audit["paired_records"],
        "unmatched_records": len(audit["unmatched_watch_record_ids"]),
        "unmatched_ids": ", ".join(audit["unmatched_watch_record_ids"]) or "none",
        "pairing_key": audit["pairing_key"],
    })
pd.DataFrame(pairing_rows)

,device,discovered_watch_records,paired_records,unmatched_records,unmatched_ids,pairing_key
0,applewatch_serie8,180,180,0,none,casefolded_relative_posix_path
1,fitbitsense2,181,180,1,st-segment/ST-m6/ST-m6_5,casefolded_relative_posix_path
2,samsunggalaxy6,179,179,0,none,casefolded_relative_posix_path
3,withingsscanwatch,180,180,0,none,casefolded_relative_posix_path


In [4]:
#| label: smartwatch-device-distribution
#| tbl-cap: Descriptive distribution across all 48 reconstruction cells; models, not records, are the rows summarized here.
device_summary = watch.groupby("device", as_index=False).agg(
    models=("model_id", "nunique"),
    paired_records=("n_paired_records", "first"),
    median_missing11_mse=("missing11_mse", "median"),
    q25_missing11_pearson=("missing11_pearson", lambda x: x.quantile(0.25)),
    median_missing11_pearson=("missing11_pearson", "median"),
    q75_missing11_pearson=("missing11_pearson", lambda x: x.quantile(0.75)),
    median_ecgfounder_probability_pearson=(
        "ecgfounder_fidelity_probability_pearson", "median"
    ),
    median_threshold_agreement=(
        "ecgfounder_fidelity_threshold_agreement", "median"
    ),
)
device_summary

,device,models,paired_records,median_missing11_mse,q25_missing11_pearson,median_missing11_pearson,q75_missing11_pearson,median_ecgfounder_probability_pearson,median_threshold_agreement
0,applewatch_serie8,48,180,0.034125,0.395037,0.422245,0.492035,0.510234,0.964889
1,fitbitsense2,48,180,0.034421,0.380306,0.417252,0.483405,0.557845,0.963926
2,samsunggalaxy6,48,179,0.035668,0.344732,0.392134,0.461917,0.450380,0.957933
3,withingsscanwatch,48,180,0.034100,0.413560,0.454914,0.499017,0.540962,0.966241


In [5]:
#| label: smartwatch-device-model-plot
#| fig-cap: Missing-11-lead correlation for all 48 model cells within each device. Each point is a model–device result, not a patient.
import plotly.express as px

fig = px.box(
    watch,
    x="device",
    y="missing11_pearson",
    points="all",
    hover_data=["model_id", "missing11_mse", "n_paired_records"],
    labels={"missing11_pearson": "Missing-11-lead Pearson", "device": "Device folder"},
)
fig.update_layout(height=500, xaxis_tickangle=-20)
fig.show()

In [6]:
#| label: smartwatch-anchor-contrast
#| tbl-cap: Locked MSE-only and full-composite anchors on each device.
anchor_ids = [
    "unet__e1c0m0d0__s42", "unet__e1c1m1d1__s42",
    "msvae__e1c0m0d0__s42", "msvae__e1c1m1d1__s42",
    "ecgaim__e1c0m0d0__s42", "ecgaim__e1c1m1d1__s42",
]
anchors = watch[watch.model_id.isin(anchor_ids)].copy()
anchors["family"] = anchors.model_id.str.split("__").str[0]
anchors["loss"] = np.where(
    anchors.model_id.str.contains("__e1c0m0d0__"), "MSE-only", "full composite"
)
anchors[[
    "device", "family", "loss", "n_paired_records",
    "missing11_mse", "missing11_pearson",
    "ecgfounder_fidelity_probability_mae",
    "ecgfounder_fidelity_probability_pearson",
    "ecgfounder_fidelity_threshold_agreement",
]].sort_values(["device", "family", "loss"])

,device,family,loss,n_paired_records,missing11_mse,missing11_pearson,ecgfounder_fidelity_probability_mae,ecgfounder_fidelity_probability_pearson,ecgfounder_fidelity_threshold_agreement
32,applewatch_serie8,ecgaim,MSE-only,180,0.034071,0.395001,0.056411,0.450581,0.965370
60,applewatch_serie8,ecgaim,full composite,180,0.035934,0.297096,0.059739,0.466839,0.961926
96,applewatch_serie8,msvae,MSE-only,180,0.027788,0.453722,0.052068,0.558638,0.969815
124,applewatch_serie8,msvae,full composite,180,0.029217,0.409025,0.043274,0.643543,0.977815
160,applewatch_serie8,unet,MSE-only,180,0.029979,0.486183,0.056604,0.554999,0.963704
188,applewatch_serie8,unet,full composite,180,0.024074,0.649695,0.052446,0.555491,0.966370
33,fitbitsense2,ecgaim,MSE-only,180,0.034085,0.392536,0.058292,0.498013,0.962926
61,fitbitsense2,ecgaim,full composite,180,0.036217,0.304211,0.061883,0.473324,0.961815
97,fitbitsense2,msvae,MSE-only,180,0.027837,0.446106,0.052405,0.587149,0.969444
125,fitbitsense2,msvae,full composite,180,0.029441,0.393680,0.045685,0.664632,0.976333


In [7]:
#| label: smartwatch-ground-truth-calibration
#| tbl-cap: Selected device-protocol calibration measurements; these are simulator/paired-reference checks, not patient outcomes.
calibration_rows = []
for device, values in payload["device_protocol_ground_truth"].items():
    calibration_rows.append({
        "device": device,
        "heart_rate_records": values["heart_rate"]["n_records"],
        "watch_vs_simulator_hr_mae_bpm": values["heart_rate"]["watch_vs_simulator"]["mae"],
        "r_wave_records": values["r_wave_amplitude"]["n_records"],
        "watch_vs_simulator_r_amp_mae_uv": values["r_wave_amplitude"]["watch_vs_simulator"]["mae"],
        "st_records": values["st_offset"]["n_records"],
        "watch_vs_simulator_st_mae_uv": values["st_offset"]["watch_vs_simulator"]["mae"],
        "square_wave_records": values["square_wave"]["n_records"],
        "max_aligned_square_wave_correlation": values["square_wave"]["watch_vs_philips_max_aligned_cross_correlation"],
    })
pd.DataFrame(calibration_rows)

,device,heart_rate_records,watch_vs_simulator_hr_mae_bpm,r_wave_records,watch_vs_simulator_r_amp_mae_uv,st_records,watch_vs_simulator_st_mae_uv,square_wave_records,max_aligned_square_wave_correlation
0,applewatch_serie8,75,5.942597,20,541.457291,80,313.955894,5,0.996773
1,fitbitsense2,76,0.221479,20,483.421251,79,370.781200,5,0.980581
2,samsunggalaxy6,75,3.626966,20,470.484465,79,298.252199,5,0.963103
3,withingsscanwatch,75,0.211283,20,588.333786,80,401.782816,5,0.901838
